In [25]:
import os
import sys
os.chdir("../..")

import duckdb
import pandas as pd
import numpy as np
from pathlib import Path

from config import DATA_ROOT, NSE_DB_PATH

In [26]:
RESEARCH_DB_PATH = DATA_ROOT / "research_v2.db"
con = duckdb.connect(RESEARCH_DB_PATH)
con.execute(f"ATTACH '{NSE_DB_PATH}' AS nse (READ_ONLY)") 

In [27]:
coverage_sql = """
WITH strike_counts AS (
    SELECT
        m.trade_date,
        i.ticker,
        i.expiry,
        DATE_DIFF('day', m.trade_date, i.expiry) AS dte,
        COUNT(DISTINCT m.instrument_key) AS n_strikes,
        SUM(m.open_interest) AS total_oi
    FROM nse.market_data_daily m
    JOIN nse.instruments i USING (instrument_key)
    WHERE i.instrument_type = 'STO'
      AND i.expiry >= m.trade_date
    GROUP BY 1,2,3
),
ranked AS (
    SELECT *,
        ROW_NUMBER() OVER (PARTITION BY ticker, trade_date ORDER BY dte ASC) AS expiry_rank
    FROM strike_counts
),
near_month AS (
    SELECT * FROM ranked WHERE expiry_rank = 1
)
SELECT
    ticker,
    COUNT(*) AS day_count,
    AVG(n_strikes) AS avg_strikes,
    AVG(total_oi) AS avg_oi,
    AVG(CAST((n_strikes >= 6 AND total_oi >= 1000) AS INT)) AS pct_usable
FROM near_month
GROUP BY 1
ORDER BY pct_usable
"""
coverage = con.execute(coverage_sql).df()
coverage.head(20)

,ticker,day_count,avg_strikes,avg_oi,pct_usable
0,CYIENT,270,55.496296,2.091736e+06,0.996296
1,ETERNAL,298,54.117450,1.014406e+08,1.000000
2,AMBER,245,61.800000,1.350472e+06,1.000000
3,HDFCBANK,498,90.646586,6.760662e+07,1.000000
4,ADANIPORTS,498,54.018072,1.442870e+07,1.000000
5,INDIGO,498,70.337349,5.562790e+06,1.000000
6,INDHOTEL,498,66.706827,1.392604e+07,1.000000
7,ASIANPAINT,498,90.620482,9.074909e+06,1.000000
8,BPCL,498,57.042169,3.668466e+07,1.000000
9,PNB,498,79.562249,1.729053e+08,1.000000


In [28]:
near_only = coverage
summary = near_only.merge(eligibility, on='ticker', how='left')
summary['coverage_ratio'] = summary['day_count'] / summary['n_days_with_fo']

print(summary['pct_usable'].describe())
print(summary['coverage_ratio'].describe())
print((summary['pct_usable'] < 0.95).sum(), "tickers below 95% usable")
summary.sort_values('pct_usable').head(15)

count    274.000000
mean       0.999986
std        0.000224
min        0.996296
25%        1.000000
50%        1.000000
75%        1.000000
max        1.000000
Name: pct_usable, dtype: float64
count    274.000000
mean       1.005609
std        0.011048
min        1.000000
25%        1.004032
50%        1.004032
75%        1.005181
max        1.111111
Name: coverage_ratio, dtype: float64
0 tickers below 95% usable


,ticker,day_count,avg_strikes,avg_oi,pct_usable,first_fo_date,last_fo_date,n_days_with_fo,coverage_ratio
0,CYIENT,270,55.496296,2.091736e+06,0.996296,2024-11-29,2025-12-30,270,1.000000
1,ETERNAL,298,54.117450,1.014406e+08,1.000000,2025-04-09,2026-06-22,296,1.006757
2,AMBER,245,61.800000,1.350472e+06,1.000000,2025-06-27,2026-06-22,243,1.008230
3,HDFCBANK,498,90.646586,6.760662e+07,1.000000,2024-06-21,2026-06-22,496,1.004032
4,ADANIPORTS,498,54.018072,1.442870e+07,1.000000,2024-06-21,2026-06-22,496,1.004032
5,INDIGO,498,70.337349,5.562790e+06,1.000000,2024-06-21,2026-06-22,496,1.004032
6,INDHOTEL,498,66.706827,1.392604e+07,1.000000,2024-06-21,2026-06-22,496,1.004032
7,ASIANPAINT,498,90.620482,9.074909e+06,1.000000,2024-06-21,2026-06-22,496,1.004032
8,BPCL,498,57.042169,3.668466e+07,1.000000,2024-06-21,2026-06-22,496,1.004032
9,PNB,498,79.562249,1.729053e+08,1.000000,2024-06-21,2026-06-22,496,1.004032


In [31]:
ghost_check_sql = """
SELECT
    i.ticker,
    i.expiry,
    MIN(m.trade_date) AS first_traded,
    MAX(m.trade_date) AS last_traded,
    COUNT(DISTINCT m.trade_date) AS n_days_traded,
    DATE_DIFF('day', MAX(m.trade_date), i.expiry) AS days_between_last_trade_and_expiry
FROM nse.instruments i
LEFT JOIN nse.market_data_daily m ON i.instrument_key = m.instrument_key
WHERE i.instrument_type = 'IDO'
GROUP BY i.ticker, i.expiry
ORDER BY i.ticker, i.expiry
"""
ghost_check = con.execute(ghost_check_sql).df()

print(ghost_check['n_days_traded'].describe())
print((ghost_check['n_days_traded'] == 0).sum(), "expiries with zero trade days")
ghost_check[ghost_check['n_days_traded'] == 0].head(20)

count    315.000000
mean      55.752381
std       58.092795
min        1.000000
25%       24.000000
50%       29.000000
75%       61.000000
max      278.000000
Name: n_days_traded, dtype: float64
0 expiries with zero trade days


,ticker,expiry,first_traded,last_traded,n_days_traded,days_between_last_trade_and_expiry


In [ ]:
# 1. The specific case you flagged
specific_sql = """
SELECT i.ticker, i.expiry, m.trade_date, m.instrument_key
FROM nse.instruments i
JOIN nse.market_data_daily m ON i.instrument_key = m.instrument_key
WHERE i.instrument_type = 'IDO' AND i.expiry = '2026-06-25'
ORDER BY i.ticker, m.trade_date
"""
print(con.execute(specific_sql).df())

# 2. Expiry weekday distribution over time, to spot the regime change
weekday_sql = """
SELECT
    i.ticker,
    i.expiry,
    DAYNAME(i.expiry) AS expiry_weekday,
    MIN(m.trade_date) AS first_traded,
    MAX(m.trade_date) AS last_traded,
    COUNT(DISTINCT m.trade_date) AS n_days_traded
FROM nse.instruments i
LEFT JOIN nse.market_data_daily m ON i.instrument_key = m.instrument_key
WHERE i.instrument_type = 'IDO'
GROUP BY i.ticker, i.expiry
ORDER BY i.expiry
"""
weekday_df = con.execute(weekday_sql).df()
print(weekday_df.groupby([weekday_df.expiry.dt.to_period('Q'), 'expiry_weekday']).size().unstack(fill_value=0))

     ticker     expiry trade_date       instrument_key
0     NIFTY 2026-06-25 2024-06-21  4761908245702378513
1     NIFTY 2026-06-25 2024-06-21   364151505584309526
2     NIFTY 2026-06-25 2024-06-21  4350736907058953251
3     NIFTY 2026-06-25 2024-06-21  1010996745537285242
4     NIFTY 2026-06-25 2024-06-21  3813444997589342054
...     ...        ...        ...                  ...
8886  NIFTY 2026-06-25 2025-07-31   709877403970023309
8887  NIFTY 2026-06-25 2025-07-31   364151505584309526
8888  NIFTY 2026-06-25 2025-07-31  5118191568898613860
8889  NIFTY 2026-06-25 2025-07-31  6895227207864269030
8890  NIFTY 2026-06-25 2025-07-31  7419336868743995477

[8891 rows x 4 columns]
expiry_weekday  Friday  Monday  Thursday  Tuesday  Wednesday
expiry                                                      
2024Q2               1       1         1        1          1
2024Q3               3      14        12       14         13
2024Q4               3       9        13       12          7
2025Q1    

In [36]:
monthly_tag_sql = """
WITH idx_expiries AS (
    SELECT DISTINCT ticker, expiry
    FROM nse.instruments
    WHERE instrument_type = 'IDO'
),
tagged AS (
    SELECT *,
        DATE_TRUNC('month', expiry) AS expiry_month,
        ROW_NUMBER() OVER (
            PARTITION BY ticker, DATE_TRUNC('month', expiry)
            ORDER BY expiry DESC
        ) AS rank_within_month
    FROM idx_expiries
)
SELECT ticker, expiry, expiry_month, (rank_within_month = 1) AS is_monthly_expiry
FROM tagged
ORDER BY ticker, expiry
"""
monthly_tags = con.execute(monthly_tag_sql).df()
monthly_tags.head(20)

,ticker,expiry,expiry_month,is_monthly_expiry
0,BANKNIFTY,2024-06-26,2024-06-01,True
1,BANKNIFTY,2024-07-03,2024-07-01,False
2,BANKNIFTY,2024-07-10,2024-07-01,False
3,BANKNIFTY,2024-07-16,2024-07-01,False
4,BANKNIFTY,2024-07-24,2024-07-01,False
5,BANKNIFTY,2024-07-31,2024-07-01,True
6,BANKNIFTY,2024-08-07,2024-08-01,False
7,BANKNIFTY,2024-08-14,2024-08-01,False
8,BANKNIFTY,2024-08-21,2024-08-01,False
9,BANKNIFTY,2024-08-28,2024-08-01,True


In [37]:
con.execute("""
    CREATE OR REPLACE TABLE near_month_options_coverage AS
    SELECT ... FROM nse.instruments i
    JOIN nse.market_data_daily m ON i.instrument_key = m.instrument_key
    ...
""")

ParserException: Parser Error: syntax error at or near ".."

LINE 3:     SELECT ... FROM nse.instruments i
                   ^